In [5]:
import pandas as pd
import numpy as np
import re

In [ ]:
df = pd.read_excel("input/combining yazar listesi.xlsx")

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

In [ ]:
df.columns = [
    "author_full_name",
    "title",
    "journal",
    "year",
    "times_cited",
    "abstract",
    "keywords",
    "affiliation",
    "author"
]

In [ ]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = text.strip()

    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
for col in df.columns:

    if df[col].dtype == "object":

        df[col] = df[col].apply(clean_text)

In [ ]:
df.isnull().sum()

In [ ]:
df["author_full_name"].nunique()

In [ ]:
df["author_full_name"].value_counts().head(20)

In [ ]:
def normalize_author_name(name):

    if pd.isna(name):
        return ""

    name = str(name)

    # Baştaki ve sondaki boşlukları sil
    name = name.strip()

    # Birden fazla boşluğu teke indir
    name = re.sub(r"\s+", " ", name)

    # Büyük-küçük harfleri düzelt
    name = name.title()

    # Virgülden önce ve sonra boşlukları düzelt
    name = re.sub(r"\s*,\s*", ", ", name)

    return name

In [ ]:
df["normalized_author"] = df["author_full_name"].apply(normalize_author_name)

In [ ]:
df[
    ["author_full_name",
     "normalized_author"]
].head(20)

In [ ]:
df["normalized_author"].nunique()

In [ ]:
import pandas as pd

def split_author(author):

    if pd.isna(author):
        return "", ""

    author = str(author).strip()

    if author == "":
        return "", ""

    if "," in author:

        parts = author.split(",", 1)

        last = parts[0].strip()

        first = parts[1].strip() if len(parts) > 1 else ""

        return last, first

    parts = author.split()

    if len(parts) == 0:
        return "", ""

    if len(parts) == 1:
        return parts[0], ""

    return parts[0], " ".join(parts[1:])

In [ ]:
split_author("Lugli, Gabriele Andrea")

In [ ]:
split_author("Chen Yang")

In [ ]:
def first_initial(first_name):

    if first_name == "":

        return ""

    return first_name[0]

In [ ]:
first_initial("Gabriele Andrea")

In [ ]:
def compare_authors(a1, a2):

    last1, first1 = split_author(a1)
    last2, first2 = split_author(a2)

    if last1 != last2:

        return False

    if first_initial(first1) != first_initial(first2):

        return False

    return True

In [ ]:
compare_authors(
    "Lugli, Gabriele Andrea",
    "Lugli, Gabriele A."
)

In [ ]:
compare_authors(
    "Smith, John",
    "Chen, Yang"
)

In [ ]:
def normalize_first_name(name):

    name = name.lower()

    name = name.replace(".", "")

    name = name.strip()

    return name

In [ ]:
normalize_first_name("Gabriele A.")

In [ ]:
from rapidfuzz import fuzz

def author_score(author1, author2):

    last1, first1 = split_author(author1)
    last2, first2 = split_author(author2)

    score = 0

    # Soyadı
    if last1.lower() == last2.lower():
        score += 40

    # İlk isim benzerliği
    score += fuzz.ratio(
        normalize_first_name(first1),
        normalize_first_name(first2)
    ) * 0.6

    return round(score,2)

In [ ]:
author_score(
    "Lugli, Gabriele Andrea",
    "Lugli, Gabriele A."
)

In [ ]:
author_score(
    "Chen, Yang",
    "Chen, Y."
)

In [ ]:
author_score(
    "Smith, John",
    "Chen, Yang"
)

In [ ]:
df["last_name"] = df["normalized_author"].apply(
    lambda x: split_author(x)[0]
)

In [ ]:
df["normalized_author"].isna().sum()

In [ ]:
(df["normalized_author"] == "").sum()

In [ ]:
df[df["normalized_author"] == ""]

In [ ]:
df = df[df["normalized_author"] != ""].copy()

In [ ]:
df.shape

In [ ]:
df["last_name"] = df["normalized_author"].apply(
    lambda x: split_author(x)[0]
)

In [ ]:
def split_author(author):

    if pd.isna(author):
        return "", ""

    author = str(author).strip()

    if author == "":
        return "", ""

    if "," in author:
        parts = author.split(",", 1)

        last = parts[0].strip()

        first = parts[1].strip() if len(parts) > 1 else ""

        return last, first

    parts = author.split()

    if len(parts) == 0:
        return "", ""

    if len(parts) == 1:
        return parts[0], ""

    return parts[0], " ".join(parts[1:])

In [ ]:
df = df[df["normalized_author"] != ""].copy()

In [ ]:
df.shape

In [ ]:
df["last_name"] = df["normalized_author"].apply(
    lambda x: split_author(x)[0]
)

In [ ]:
last_name_groups = df.groupby("last_name")

In [ ]:
# =====================================================
# AUTHOR NORMALIZATION
# =====================================================

# Bibliometric Normalization System (BNS)

## Module 03 - Author Normalization

**Objective**

This notebook identifies and merges author name variants in the Web of Science dataset.

Outputs:

- Merged_Author_Table.xlsx
- Normalized_WoS_Data.xlsx

In [ ]:
author_groups = {}

for last_name, group in df.groupby("last_name"):
    author_groups[last_name] = (
        group["normalized_author"]
        .dropna()
        .unique()
        .tolist()
    )

In [ ]:
df["last_name"] = df["normalized_author"].apply(
    lambda x: split_author(x)[0]
)

In [ ]:
from rapidfuzz import fuzz

def author_name_similarity(author1, author2):

    author1 = str(author1).strip().lower()
    author2 = str(author2).strip().lower()

    return fuzz.token_sort_ratio(author1, author2)

In [ ]:
author_name_similarity(
    "Lugli, Gabriele Andrea",
    "Lugli, Gabriele A."
)

In [ ]:
def find_candidates(author_list, threshold=90):

    candidates = []

    for i in range(len(author_list)):

        for j in range(i + 1, len(author_list)):

            score = author_name_similarity(
                author_list[i],
                author_list[j]
            )

            if score >= threshold:

                candidates.append({
                    "author_1": author_list[i],
                    "author_2": author_list[j],
                    "score": score
                })

    return candidates

In [ ]:
first_lastname = list(author_groups.keys())[0]

print(first_lastname)

In [ ]:
candidate_pairs = find_candidates(
    author_groups[first_lastname]
)

candidate_pairs

In [ ]:
largest_group = max(
    author_groups.items(),
    key=lambda x: len(x[1])
)

largest_lastname = largest_group[0]

print(largest_lastname)
print(len(largest_group[1]))

## 5.1 Affiliation Standardization

In [ ]:
def normalize_affiliation(aff):

    if pd.isna(aff):
        return ""

    aff = str(aff).lower()

    aff = aff.replace("univ.", "university")
    aff = aff.replace("dept.", "department")
    aff = aff.replace("&", "and")

    aff = " ".join(aff.split())

    return aff

In [ ]:
df["normalized_affiliation"] = (
    df["affiliation"]
    .apply(normalize_affiliation)
)

In [ ]:
df[[
    "affiliation",
    "normalized_affiliation"
]].head()

## 5.2 Keyword Standardization

In [ ]:
def normalize_keywords(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = text.replace(";", ",")

    keywords = [
        x.strip()
        for x in text.split(",")
        if x.strip()
    ]

    keywords = sorted(set(keywords))

    return ",".join(keywords)

In [ ]:
df["normalized_keywords"] = (
    df["keywords"]
    .apply(normalize_keywords)
)

In [ ]:
df[[
    "keywords",
    "normalized_keywords"
]].head()

BÖLÜM 3 — Publication Year

In [ ]:
df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)

In [ ]:
df["year"].describe()

BÖLÜM 4 — Name Similarity

In [ ]:
from rapidfuzz import fuzz

def name_similarity(a, b):

    return fuzz.token_sort_ratio(
        str(a),
        str(b)
    )

BÖLÜM 5 — Affiliation Similarity

In [ ]:
def affiliation_similarity(a, b):

    return fuzz.token_sort_ratio(
        str(a),
        str(b)
    )

BÖLÜM 6 — Keyword Similarity

In [ ]:
def keyword_similarity(a, b):

    return fuzz.token_sort_ratio(
        str(a),
        str(b)
    )

BÖLÜM 7 — Year Similarity

In [ ]:
def year_similarity(y1, y2):

    if pd.isna(y1) or pd.isna(y2):
        return 50

    diff = abs(y1 - y2)

    if diff == 0:
        return 100

    if diff <= 2:
        return 80

    if diff <= 5:
        return 60

    return 20

## 5.3 Author Confidence Score

In [ ]:
def confidence_score(
    author1,
    author2,
    aff1,
    aff2,
    key1,
    key2,
    year1,
    year2
):

    name_score = name_similarity(author1, author2)

    aff_score = affiliation_similarity(aff1, aff2)

    keyword_score = keyword_similarity(key1, key2)

    year_score = year_similarity(year1, year2)

    final_score = (
        name_score * 0.50
        + aff_score * 0.25
        + keyword_score * 0.15
        + year_score * 0.10
    )

    return {
        "name": round(name_score,2),
        "affiliation": round(aff_score,2),
        "keywords": round(keyword_score,2),
        "year": round(year_score,2),
        "confidence": round(final_score,2)
    }

In [ ]:
confidence_score(

    "Lugli, Gabriele Andrea",
    "Lugli, Gabriele A.",

    "University of Parma",
    "Univ. of Parma",

    "microbiome, probiotics",
    "probiotics, microbiome",

    2022,
    2023
)

In [ ]:
df.head(10)

In [ ]:
row1 = df.iloc[0]
row2 = df.iloc[1]

In [ ]:
confidence_score(

    row1["normalized_author"],
    row2["normalized_author"],

    row1["normalized_affiliation"],
    row2["normalized_affiliation"],

    row1["normalized_keywords"],
    row2["normalized_keywords"],

    row1["year"],
    row2["year"]

)

In [ ]:
def evaluate_pair(row1,row2):

    score = confidence_score(

        row1["normalized_author"],
        row2["normalized_author"],

        row1["normalized_affiliation"],
        row2["normalized_affiliation"],

        row1["normalized_keywords"],
        row2["normalized_keywords"],

        row1["year"],
        row2["year"]

    )

    return score

In [ ]:
evaluate_pair(
    df.iloc[0],
    df.iloc[1]
)

In [ ]:
def merge_decision(score):

    if score["confidence"] >= 95:
        return "Automatic Merge"

    elif score["confidence"] >= 85:
        return "Manual Review"

    else:
        return "Different Authors"

In [ ]:
score = evaluate_pair(
    df.iloc[0],
    df.iloc[1]
)

merge_decision(score)

# 6. Author Merge Engine

In [ ]:
merge_table = []

In [ ]:
for lastname in author_groups:

    print(lastname)

In [ ]:
for lastname in author_groups:

    authors = author_groups[lastname]

    print(lastname, len(authors))

In [ ]:
for lastname in author_groups:

    authors = author_groups[lastname]

    if len(authors) < 2:
        continue

    print(lastname)

In [ ]:
merge_table = []

for lastname in author_groups:

    authors = author_groups[lastname]

    if len(authors) < 2:
        continue

    for i in range(len(authors)):

        for j in range(i+1, len(authors)):

            merge_table.append({

                "lastname": lastname,

                "author1": authors[i],

                "author2": authors[j]

            })

In [ ]:
len(merge_df)

In [ ]:
author_lookup = (
    df
    .drop_duplicates("normalized_author")
    .set_index("normalized_author", drop=False)
)

In [ ]:
author_lookup.head()

In [ ]:
scores = []

for _, row in merge_df.iterrows():

    try:

        a = author_lookup.loc[row["author1"]]
        b = author_lookup.loc[row["author2"]]

        result = confidence_score(

            a["normalized_author"],
            b["normalized_author"],

            a["normalized_affiliation"],
            b["normalized_affiliation"],

            a["normalized_keywords"],
            b["normalized_keywords"],

            a["year"],
            b["year"]

        )

        scores.append(result["confidence"])

    except Exception as e:

        print("HATA:")
        print(row["author1"])
        print(row["author2"])
        print(e)

        break

In [ ]:
drop=False

In [ ]:
scores = []

for _, row in merge_df.iterrows():

    a = author_lookup.loc[row["author1"]]
    b = author_lookup.loc[row["author2"]]

    result = confidence_score(

        a["normalized_author"],
        b["normalized_author"],

        a["normalized_affiliation"],
        b["normalized_affiliation"],

        a["normalized_keywords"],
        b["normalized_keywords"],

        a["year"],
        b["year"]

    )

    scores.append(result["confidence"])

# 7. Canonical Author Selection

In [ ]:
automatic_merge = merge_df[
    merge_df["decision"] == "Automatic Merge"
].copy()

automatic_merge.head()

In [ ]:
len(automatic_merge)

In [ ]:
def choose_canonical_name(author1, author2):

    # Daha uzun isim tercih edilir
    if len(author1) >= len(author2):
        return author1
    else:
        return author2

In [ ]:
automatic_merge["canonical_author"] = automatic_merge.apply(
    lambda row: choose_canonical_name(
        row["author1"],
        row["author2"]
    ),
    axis=1
)

In [ ]:
automatic_merge[
    ["author1", "author2", "canonical_author", "confidence"]
].head(20)

In [ ]:
merge_df.columns.tolist()

In [ ]:
author_lookup.head()

In [ ]:
author_lookup.columns.tolist()

In [ ]:
row = merge_df.iloc[0]

row

In [ ]:
a = author_lookup.loc[row["author1"]]
b = author_lookup.loc[row["author2"]]

print(a["normalized_author"])
print(b["normalized_author"])

In [ ]:
confidence_score(

    a["normalized_author"],
    b["normalized_author"],

    a["normalized_affiliation"],
    b["normalized_affiliation"],

    a["normalized_keywords"],
    b["normalized_keywords"],

    a["year"],
    b["year"]
)

# =====================================================
# Bibliometric Normalization System (BNS)
# Module 04 - Merge Engine
# Version : 1.0
# Author  : <Samet UÇAK>
# =====================================================

# Bibliometric Normalization System (BNS)

## Module 04 — Merge Engine

Bu modül aynı araştırmacıya ait isim varyasyonlarını tespit eder, güven puanını hesaplar ve Canonical Author tablosunu oluşturur.

In [ ]:
print("="*70)
print("BIBLIOMETRIC NORMALIZATION SYSTEM")
print("MODULE 04 - MERGE ENGINE")
print("="*70)

In [ ]:
required_columns = [
    "normalized_author",
    "last_name",
    "normalized_affiliation",
    "normalized_keywords",
    "year"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    print("Eksik sütunlar:")
    print(missing)
else:
    print("✓ Tüm gerekli sütunlar mevcut.")

In [ ]:
author_lookup = (
    df
    .drop_duplicates("normalized_author")
    .set_index("normalized_author", drop=False)
)

In [ ]:
print(author_lookup.shape)

In [ ]:
merge_table = []

for lastname, authors in author_groups.items():

    if len(authors) < 2:
        continue

    for i in range(len(authors)):

        for j in range(i+1, len(authors)):

            merge_table.append({

                "lastname": lastname,

                "author1": authors[i],

                "author2": authors[j]

            })

In [ ]:
merge_df = pd.DataFrame(merge_table)

print(merge_df.shape)
merge_df.head()

In [ ]:
merge_df["confidence"] = merge_df.apply(
    calculate_confidence,
    axis=1
)

In [ ]:
calculate_confidence

In [ ]:
def calculate_confidence(row):

    a = author_lookup.loc[row["author1"]]
    b = author_lookup.loc[row["author2"]]

    result = confidence_score(

        a["normalized_author"],
        b["normalized_author"],

        a["normalized_affiliation"],
        b["normalized_affiliation"],

        a["normalized_keywords"],
        b["normalized_keywords"],

        a["year"],
        b["year"]

    )

    return result["confidence"]

In [ ]:
calculate_confidence(merge_df.iloc[0])

In [ ]:
merge_df["confidence"] = merge_df.apply(
    calculate_confidence,
    axis=1
)

In [ ]:
print("confidence_score:", callable(confidence_score))
print("merge_decision:", callable(merge_decision))

In [ ]:
def calculate_confidence(row):

    author1 = row["author1"]
    author2 = row["author2"]

    a = author_lookup.loc[author1]
    b = author_lookup.loc[author2]

    score = confidence_score(

        a["normalized_author"],
        b["normalized_author"],

        a["normalized_affiliation"],
        b["normalized_affiliation"],

        a["normalized_keywords"],
        b["normalized_keywords"],

        a["year"],
        b["year"]

    )

    return score["confidence"]

In [ ]:
test_row = merge_df.iloc[0]

calculate_confidence(test_row)

In [ ]:
merge_df["confidence"] = merge_df.apply(
    calculate_confidence,
    axis=1
)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
print("✓ similarity.py başarıyla yüklendi.")

In [ ]:
confidence_score(

    "Lugli, Gabriele Andrea",

    "Lugli, Gabriele A.",

    "University of Parma",

    "University of Parma",

    "microbiome, probiotics",

    "probiotics, microbiome",

    2022,

    2023

)

In [ ]:
from src.merge_engine import (
    build_merge_candidates,
    score_merge_candidates
)

In [ ]:
merge_df = build_merge_candidates(author_groups)

merge_df.head()

In [ ]:
merge_df.shape

In [ ]:
merge_df = score_merge_candidates(

    merge_df,

    author_lookup

)

In [ ]:
merge_df.head()

In [ ]:
from src.merge_engine import (
    build_merge_candidates,
    score_merge_candidates
)

In [ ]:
from src.merge_engine import select_canonical_author

MODÜL 04 – BÖLÜM 2: Canonical Author Mapping

In [ ]:
automatic_merge = merge_df[
    merge_df["decision"] == "Automatic Merge"
].copy()

print(f"Otomatik birleşecek kayıt sayısı: {len(automatic_merge)}")
automatic_merge.head()

In [ ]:
print(df.shape)

In [ ]:
print(author_groups is not None)

In [ ]:
print(author_lookup.shape)

In [ ]:
from src.merge_engine import (
    build_merge_candidates,
    score_merge_candidates
)

merge_df = build_merge_candidates(author_groups)

print(merge_df.shape)
merge_df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
author_lookup = (
    df
    .sort_values("times_cited", ascending=False)
    .drop_duplicates(subset="normalized_author")
    .set_index("normalized_author")
)

print(author_lookup.shape)
author_lookup.head()

In [ ]:
merge_df = build_merge_candidates(author_groups)

merge_df = score_merge_candidates(
    merge_df,
    author_lookup
)

merge_df.head()

In [ ]:
import src.merge_engine as me

print(me.__file__)

In [ ]:
import src.similarity as sim

print(dir(sim))

In [ ]:
import src.merge_engine as me

print(me.__file__)
print("confidence_score" in dir(me))
print(dir(me))

In [ ]:
import src.merge_engine as me

print(me.confidence_score)

In [3]:
importlib.reload(sim)
importlib.reload(me)

print("✓ similarity.py yüklendi")
print("✓ merge_engine.py yüklendi")

✓ similarity.py yüklendi
✓ merge_engine.py yüklendi
